# Milestone 1 — Colab pipeline (7 code cells)

**One repository:** code, Colab notebook, diagnostics and Jupyter Book all stay in ahtuan171/arXiv-Project. Persisted crawl data stays on Google Drive, not in Git.

Run code cells 1 through 7 in order. Source and raw-reference requests are disabled by default. All exports are separately disabled by default. To test a small download, edit the flags in cell 2 and rerun the relevant cells. Run cell 6 after any data writes and before cell 7. Do not run concurrent writers to the same Drive folder.

1. Setup: Drive mount, code clone/pull on local Colab disk, Python dependencies
2. Configuration: ID range, API key, network/exports, bounded limits
3. Preflight: fast metadata checks and optional offline pytest
4. Source downloader: versioned vN, resume, progress bar
5. Raw references: S2 publication-level citations including non-arXiv
6. Validation: one full SHA-256 source pass plus metadata/ledger integrity
7. Summary: verified coverage, missing evidence, opt-in JSON report

Legacy comparison and the old four CSV tables belong to the separate read-only diagnostics notebook in this **same repo**: notebooks/m1_legacy_debug.ipynb. The old notebook workflow is preserved in Git history, not executed by the primary notebook.

A source version marked pending or failed is NOT counted as downloaded. An offline test pass is NOT evidence of a complete crawl. A historical benchmark from the older pipeline is NOT a measured benchmark of this version-aware pipeline.


## 1. Environment Setup
Mount Drive, update code on /content, install dependencies. No Git operations inside Drive.

In [ ]:
# 1/7 SETUP: clone code locally, keep all data on Drive
import os, sys, subprocess
from pathlib import Path

BRANCH = "feat/milestone1-compliance"
try:
    from google.colab import drive
except ImportError:
    CODE_DIR = Path.cwd().resolve()
    DATA_DIR = Path(os.environ.get("ARXIV_DATA_DIR", CODE_DIR / "data")).resolve()
else:
    drive.mount("/content/drive")
    CODE_DIR = Path("/content/arxiv-code")
    DATA_DIR = Path(os.environ.get(
        "ARXIV_DATA_DIR",
        "/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data",
    )).resolve()
    if not (CODE_DIR / ".git").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1", "--single-branch",
            "--branch", BRANCH,
            "https://github.com/ahtuan171/arXiv-Project.git", str(CODE_DIR),
        ], check=True)
    else:
        actual = subprocess.check_output(
            ["git", "-C", str(CODE_DIR), "branch", "--show-current"], text=True
        ).strip()
        if actual != BRANCH:
            raise RuntimeError("Wrong branch; restart runtime and re-open the GitHub notebook")
        subprocess.run(
            ["git", "-C", str(CODE_DIR), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )

if not (DATA_DIR / "papers.jsonl").is_file():
    raise FileNotFoundError(f"Missing original metadata: {DATA_DIR / 'papers.jsonl'}")
os.environ["ARXIV_DATA_DIR"] = str(DATA_DIR)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
if "src.config" in sys.modules:
    from src import config
    if Path(config.DEFAULT_DATA_DIR).resolve() != DATA_DIR:
        raise RuntimeError("Old src.config imported; use Runtime -> Restart session")
else:
    from src import config
assert Path(config.DEFAULT_DATA_DIR).resolve() == DATA_DIR
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")],
    check=True,
)
print("✅ Environment ready")
print("Code:", CODE_DIR, "| Data:", DATA_DIR)
subprocess.run(["git", "-C", str(CODE_DIR), "log", "-1", "--oneline"], check=True)


## 2. Configuration
All safety flags live here. By default, no crawler or exporter writes new dataset files.

In [ ]:
# 2/7 CONFIG: edit parameters ONLY here
start_id = "2502.13235"
end_id = "2502.13245"
LIMIT = 0                         # 0 = all IDs in selected range

RUN_OFFLINE_TESTS = True
DOWNLOAD_VERSIONS = False         # Default: no source requests/writes
VERSION_LIMIT = 3                 # Max NETWORK ATTEMPTS per run
VERSION_RETRY_ERRORS = False
VERSION_RETRY_NOT_FOUND = False

FETCH_RAW_REFERENCES = False      # Default: no S2 requests/writes
RAW_REF_LIMIT = 3                 # Max papers queried per run
RAW_REF_RETRY_ERRORS = False
RAW_REF_RETRY_NOT_FOUND = False

EXPORT_VERSION_CSV = False
EXPORT_VERSION_METADATA = False
EXPORT_M1_INTEGRITY_REPORT = False
EXPORT_M1_REPORT = False
AUDIT_SAMPLE_SIZE = 10

from src import config
from src.utils import parse_arxiv_range
id_list = parse_arxiv_range(start_id, end_id)
if LIMIT > 0:
    id_list = id_list[:LIMIT]
if not id_list:
    raise ValueError("Empty ID range")
if VERSION_LIMIT < 1 or RAW_REF_LIMIT < 1:
    raise ValueError("VERSION_LIMIT and RAW_REF_LIMIT must be >= 1")

try:
    from google.colab import userdata
    secret = userdata.get("S2_API_KEY") or ""
except Exception:
    secret = os.environ.get("S2_API_KEY", "")
if secret.strip():
    config.S2_API_KEY = secret.strip()
    os.environ["S2_API_KEY"] = secret.strip()
print(f"✅ Config: {len(id_list)} IDs ({id_list[0]} to {id_list[-1]})")
print("Semantic Scholar:", "API key ready" if config.S2_API_KEY else "anonymous")
print("Source:", "LIVE" if DOWNLOAD_VERSIONS else "DRY-RUN",
      "| Raw references:", "LIVE" if FETCH_RAW_REFERENCES else "DRY-RUN")
VALIDATION_REQUIRED = True


## 3. Preflight and Offline Tests
Metadata-only inspection and independent pytest fixtures. No arXiv or Semantic Scholar calls.

In [ ]:
# 3/7 PREFLIGHT: metadata-only audit and optional OFFLINE tests
from src.scripts.audit_missing_versions import audit_missing_versions

preflight = audit_missing_versions(
    config.DEFAULT_DATA_DIR, ids=id_list, sample_size=AUDIT_SAMPLE_SIZE
)
print("=== METADATA PREFLIGHT (READ-ONLY) ===")
for key, val in preflight["summary"].items():
    print(f"{key:31}: {val}")
for warning in preflight.get("warnings", [])[:AUDIT_SAMPLE_SIZE]:
    print("Metadata warning:", warning)

if RUN_OFFLINE_TESTS:
    OFFLINE_TESTS = [
        "tests/test_version_downloader.py",
        "tests/test_audit_version_downloads.py",
        "tests/test_audit_legacy_matches.py",
        "tests/test_audit_progress.py",
        "tests/test_build_version_metadata.py",
        "tests/test_raw_reference_collector.py",
        "tests/test_audit_m1_integrity.py",
        "tests/test_m1_offline_end_to_end.py",
        "tests/test_build_m1_report.py",
        "tests/test_notebook_layout.py",
    ]
    assert all((CODE_DIR / path).is_file() for path in OFFLINE_TESTS)
    child_env = dict(os.environ, ARXIV_DATA_DIR="/tmp/m1-fixtures-only",
                     S2_API_KEY="", PYTHONDONTWRITEBYTECODE="1",
                     PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
    subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", *OFFLINE_TESTS],
        cwd=str(CODE_DIR), env=child_env, check=True,
    )
    print("✅ Offline tests passed.")
else:
    print("Offline tests skipped.")


## 4. Download Versions
Request source for specific revisions only when explicitly enabled; cap counts attempts.

In [ ]:
# 4/7 SOURCE VERSIONS: explicit network opt-in, bounded attempts
from tqdm.auto import tqdm
from src.download_module.version_downloader import run_version_downloads

if DOWNLOAD_VERSIONS:
    VALIDATION_REQUIRED = True
with tqdm(desc="Source vN", unit="attempt", dynamic_ncols=True,
          disable=not DOWNLOAD_VERSIONS) as bar:
    def on_source_progress(done, total, event):
        if bar.total != total:
            bar.total = total
            bar.refresh()
        bar.update(done - bar.n)
        bar.set_postfix_str(event["status"])
    version_result = run_version_downloads(
        config.DEFAULT_DATA_DIR, ids=id_list,
        max_downloads=VERSION_LIMIT, dry_run=not DOWNLOAD_VERSIONS,
        retry_errors=VERSION_RETRY_ERRORS,
        retry_not_found=VERSION_RETRY_NOT_FOUND,
        progress_callback=on_source_progress if DOWNLOAD_VERSIONS else None,
    )
print("=== VERSION DOWNLOAD ===")
print("Plan:", version_result["plan"]["summary"])
print("Result:", version_result["summary"])
if not DOWNLOAD_VERSIONS:
    print("✅ DRY-RUN: no files downloaded.")


## 5. Raw References
Retain all S2 entries, including those without an arXiv ID. Publication-level only.

In [ ]:
# 5/7 RAW REFERENCES: keep non-arXiv entries, separate ledger
from tqdm.auto import tqdm
from src.crawl_references_module.raw_reference_collector import collect_raw_references

if FETCH_RAW_REFERENCES:
    VALIDATION_REQUIRED = True
with tqdm(desc="Raw references", unit="paper", dynamic_ncols=True,
          disable=not FETCH_RAW_REFERENCES) as bar:
    def on_raw_progress(done, total, event):
        if bar.total != total:
            bar.total = total
            bar.refresh()
        bar.update(done - bar.n)
        bar.set_postfix_str(event["status"])
    raw_ref_result = collect_raw_references(
        config.DEFAULT_DATA_DIR, ids=id_list,
        max_papers=RAW_REF_LIMIT, dry_run=not FETCH_RAW_REFERENCES,
        retry_errors=RAW_REF_RETRY_ERRORS,
        retry_not_found=RAW_REF_RETRY_NOT_FOUND,
        progress_callback=on_raw_progress if FETCH_RAW_REFERENCES else None,
    )
print("=== RAW REFERENCES ===")
print(raw_ref_result["summary"])
if not FETCH_RAW_REFERENCES:
    print("✅ DRY-RUN: no Semantic Scholar requests.")


## 6. Validation and Finalization
Hash each saved revision once, reuse the result for the integrity audit, export only if enabled.

In [ ]:
# 6/7 VERIFY ONCE: SHA-256, metadata and raw ledger, exports opt-in
from tqdm.auto import tqdm
from src.scripts.audit_version_downloads import audit_version_downloads, export_version_csv
from src.scripts.build_version_metadata import build_version_metadata, export_version_metadata
from src.scripts.audit_m1_integrity import audit_m1_integrity, export_integrity_report

with tqdm(desc="Verify version hashes", unit="version", dynamic_ncols=True) as bar:
    def on_verify(done, total, record):
        if bar.total != total:
            bar.total = total
            bar.refresh()
        bar.update(done - bar.n)
        bar.set_postfix_str(record["status"])
    version_audit = audit_version_downloads(
        config.DEFAULT_DATA_DIR, sample_size=AUDIT_SAMPLE_SIZE,
        progress_callback=on_verify,
    )
metadata_report = build_version_metadata(
    config.DEFAULT_DATA_DIR, sample_size=AUDIT_SAMPLE_SIZE,
)
# Update requested snapshots BEFORE validating snapshot freshness.
if EXPORT_VERSION_CSV:
    print("Version CSV:", export_version_csv(version_audit))
if EXPORT_VERSION_METADATA:
    print("Metadata index:", export_version_metadata(metadata_report))

integrity_report = audit_m1_integrity(
    config.DEFAULT_DATA_DIR, sample_size=AUDIT_SAMPLE_SIZE,
    version_report=version_audit, metadata_report=metadata_report,
)
VALIDATION_REQUIRED = False
print("=== INTEGRITY:", integrity_report["status"], "===")
print("Coverage:", integrity_report["coverage"])
print("Sources:", integrity_report["source_versions"])
print("Raw citations:", integrity_report["raw_references"])
for label in ("blockers", "incomplete", "warnings"):
    for item in integrity_report[label][:AUDIT_SAMPLE_SIZE]:
        print(label.upper(), ":", item)
if EXPORT_M1_INTEGRITY_REPORT:
    print("Integrity JSON:", export_integrity_report(integrity_report))
else:
    print("✅ Integrity report in memory only.")


## 7. Summary and Report
Reuse cell 6. Metrics without actual Colab evidence remain unavailable.

In [ ]:
# 7/7 SUMMARY: reuse recent verification, no repeated hash scan
from src.scripts.build_m1_report import build_m1_report, export_m1_report
from IPython.display import display
import pandas as pd

if "integrity_report" not in globals() or VALIDATION_REQUIRED:
    raise RuntimeError("Run cell 6 again after the latest data change.")
m1_summary = build_m1_report(
    config.DEFAULT_DATA_DIR, integrity_report=integrity_report,
)
c = m1_summary["coverage"]
display(pd.DataFrame([
    {"Metric": "Valid metadata papers", "Value": c["valid_metadata_papers"]},
    {"Metric": "Expected source versions", "Value": c["expected_versions"]},
    {"Metric": "Verified source versions", "Value": c["verified_source_versions"]},
    {"Metric": "Version coverage (%)", "Value": c["source_coverage_pct"]},
    {"Metric": "Raw reference papers verified", "Value": c["raw_reference_papers_verified"]},
]))
print("Integrity status:", m1_summary["integrity_status"])
print("Source size summary:", m1_summary["source_file_statistics"]
      or m1_summary["source_statistics_unavailable_reason"])
print("Official pipeline benchmark:",
      m1_summary["benchmark_evidence"]["official_version_pipeline_metrics"])
print("Outstanding evidence:")
for item in m1_summary["missing_evidence"]:
    print(" -", item)
if EXPORT_M1_REPORT:
    print("Summary JSON:", export_m1_report(m1_summary))
else:
    print("✅ Summary is read-only.")
